Hybrid Search RAG Using Pinecone

In [1]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

#set up environment
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
api_key=os.getenv("PINECONE_API_KEY")

In [3]:
import os
from pinecone import Pinecone, ServerlessSpec

index_name="hybrid-search-langchain-pinecone"

#intilize pinecone client
pc=Pinecone(api_key=api_key)

#create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, #dimensone of dense vector ->it is 384 we use huggingface embedding which default converts any text in 384 dimenson vector
        metric='dotproduct',
        spec=ServerlessSpec(cloud='aws',region='us-east-1')

    )

In [4]:
index=pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-xnexi8r.svc.aped-4627-b74a.pinecone.io')

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

c:\Users\Vansh Parmar\Documents\Generative_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4211.52it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
from pinecone_text.sparse import BM25Encoder
#this encode uses TF-IDF technique by default

bm25_encoder=BM25Encoder().default()
bm25_encoder


In [7]:
sentences=[
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

bm25_encoder.fit(sentences)

#store values in to json file
bm25_encoder.dump("bm25_values.json")

bm25_encoder=BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 57.08it/s]


In [8]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001F388F27CB0>, index=Index(host='https://hybrid-search-langchain-pinecone-xnexi8r.svc.aped-4627-b74a.pinecone.io'))

In [ ]:
import inspect
from langchain_community.retrievers import pinecone_hybrid_search

print(inspect.getsource(pinecone_hybrid_search.create_index))
#update in .venv\Lib\site-packages\langchain_community\retrievers\pinecone_hybrid_search.py
#in above file if error in 10th block
#replace index.upsert(vectors, namespace=namespace) by 
# index.upsert(
#     vectors=vectors,
#     namespace=namespace
# )

def create_index(
    contexts: List[str],
    index: Any,
    embeddings: Embeddings,
    sparse_encoder: Any,
    ids: Optional[List[str]] = None,
    metadatas: Optional[List[dict]] = None,
    namespace: Optional[str] = None,
    text_key: str = "context",
) -> None:
    """Create an index from a list of contexts.

    It modifies the index argument in-place!

    Args:
        contexts: List of contexts to embed.
        index: Index to use.
        embeddings: Embeddings model to use.
        sparse_encoder: Sparse encoder to use.
        ids: List of ids to use for the documents.
        metadatas: List of metadata to use for the documents.
        namespace: Namespace value for index partition.
    """
    batch_size = 32
    _iterator = range(0, len(contexts), batch_size)
    try:
        from tqdm.auto import tqdm

        _iterator = tqdm(_iterator)
    except ImportError:
        pass

    if ids is None:
        # create unique ids using hash of the text
        ids = [has

In [10]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]
)

100%|██████████| 1/1 [00:02<00:00,  2.34s/it]


In [11]:
retriever.invoke("What city did i visit last?")

[Document(metadata={'score': 0.287524819}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.259537339}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.235914916}, page_content='In 2023, I visited Paris')]